# 09 - Presentation Plot Production

This notebook reads the active Google Drive result structure for the AMR project and generates fixed-name visual assets for the final 10-slide presentation. It compares AWGN baselines, models trained from scratch on fading datasets, and AWGN-pretrained models fine-tuned on fading datasets.

Expected Drive root: `/content/drive/MyDrive/AMR-Project/`

Output folder: `presentation_assets/`

In [ ]:
# Environment setup
import os
import csv
import gc
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

COLAB_PROJECT_DIR = Path('/content/drive/MyDrive/AMR-Project')
PROJECT_DIR = COLAB_PROJECT_DIR if COLAB_PROJECT_DIR.exists() else Path.cwd().resolve()
RESULTS_DIR = PROJECT_DIR / 'results'
FINETUNE_DIR = PROJECT_DIR / 'fine_tuning_results'
ASSET_DIR = PROJECT_DIR / 'presentation_assets'
ASSET_DIR.mkdir(parents=True, exist_ok=True)

print('Project dir:', PROJECT_DIR)
print('Results dir:', RESULTS_DIR)
print('Fine-tuning dir:', FINETUNE_DIR)
print('Output assets:', ASSET_DIR)


In [ ]:
# Experiment configuration
MODELS = ['MCLDNN', 'PET-CGDNN']
CHANNELS = ['Rayleigh', 'Rician K=3', 'Rician K=10']
ALL_CHANNELS = ['AWGN'] + CHANNELS
SNRS = list(range(-20, 20, 2))

MODEL_COLORS = {
    'MCLDNN': '#2563eb',
    'PET-CGDNN': '#dc2626',
}
CHANNEL_COLORS = {
    'AWGN': '#111827',
    'Rayleigh': '#7c3aed',
    'Rician K=3': '#059669',
    'Rician K=10': '#ea580c',
}
STRATEGY_COLORS = {
    'baseline_awgn': '#111827',
    'fading_trained': '#0f766e',
    'fine_tuned': '#b45309',
}
STRATEGY_LABELS = {
    'baseline_awgn': 'AWGN baseline',
    'fading_trained': 'Fading-trained',
    'fine_tuned': 'Fine-tuned',
}

EXPERIMENTS = [
    {'strategy': 'baseline_awgn', 'model': 'MCLDNN', 'channel': 'AWGN', 'path': RESULTS_DIR / 'mcldnn_awgn'},
    {'strategy': 'baseline_awgn', 'model': 'PET-CGDNN', 'channel': 'AWGN', 'path': RESULTS_DIR / 'petcgdnn_awgn'},
    {'strategy': 'fading_trained', 'model': 'MCLDNN', 'channel': 'Rayleigh', 'path': RESULTS_DIR / 'mcldnn_rayleigh'},
    {'strategy': 'fading_trained', 'model': 'PET-CGDNN', 'channel': 'Rayleigh', 'path': RESULTS_DIR / 'petcgdnn_rayleigh'},
    {'strategy': 'fading_trained', 'model': 'MCLDNN', 'channel': 'Rician K=3', 'path': RESULTS_DIR / 'mcldnn_rician_K3'},
    {'strategy': 'fading_trained', 'model': 'PET-CGDNN', 'channel': 'Rician K=3', 'path': RESULTS_DIR / 'petcgdnn_rician_K3'},
    {'strategy': 'fading_trained', 'model': 'MCLDNN', 'channel': 'Rician K=10', 'path': RESULTS_DIR / 'mcldnn_rician_K10'},
    {'strategy': 'fading_trained', 'model': 'PET-CGDNN', 'channel': 'Rician K=10', 'path': RESULTS_DIR / 'petcgdnn_rician_K10'},
    {'strategy': 'fine_tuned', 'model': 'MCLDNN', 'channel': 'Rayleigh', 'path': FINETUNE_DIR / 'mcldnn_rayleigh'},
    {'strategy': 'fine_tuned', 'model': 'PET-CGDNN', 'channel': 'Rayleigh', 'path': FINETUNE_DIR / 'petcgdnn_rayleigh'},
    {'strategy': 'fine_tuned', 'model': 'MCLDNN', 'channel': 'Rician K=3', 'path': FINETUNE_DIR / 'mcldnn_rician_k3'},
    {'strategy': 'fine_tuned', 'model': 'PET-CGDNN', 'channel': 'Rician K=3', 'path': FINETUNE_DIR / 'petcgdnn_rician_k3'},
    {'strategy': 'fine_tuned', 'model': 'MCLDNN', 'channel': 'Rician K=10', 'path': FINETUNE_DIR / 'mcldnn_rician_k10'},
    {'strategy': 'fine_tuned', 'model': 'PET-CGDNN', 'channel': 'Rician K=10', 'path': FINETUNE_DIR / 'petcgdnn_rician_k10'},
]

DATASET_FILES = {
    'AWGN': PROJECT_DIR / 'RML2016.10a_dict.pkl',
    'Rayleigh': PROJECT_DIR / 'RML2016.10a_rayleigh.pkl',
    'Rician K=3': PROJECT_DIR / 'RML2016.10a_rician_K3.pkl',
    'Rician K=10': PROJECT_DIR / 'RML2016.10a_rician_K10.pkl',
}


In [ ]:
# Utility functions
def asset_path(name):
    return ASSET_DIR / name

def load_pickle(path):
    if path is None or not Path(path).exists():
        return None
    with open(path, 'rb') as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0)
            return pickle.load(f, encoding='iso-8859-1')

def find_metric_file(exp_path, filename):
    candidates = [exp_path / filename, exp_path / 'predictions' / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

def find_weight_files(exp_path):
    files = []
    for folder in [exp_path, exp_path / 'weights']:
        if folder.exists():
            files.extend(sorted([p.name for p in folder.glob('*.h5')]))
            files.extend(sorted([p.name for p in folder.glob('*.keras')]))
    return files

def normalize_metric_dict(metric):
    if metric is None:
        return None
    return {int(k): float(v) for k, v in metric.items()}

def mean_for_snr(metric, predicate):
    if not metric:
        return np.nan
    values = [value for snr, value in sorted(metric.items()) if predicate(snr)]
    return float(np.mean(values)) if values else np.nan

def safe_pct(value):
    return 'NA' if value is None or np.isnan(value) else f'{100.0 * value:.2f}'

def save_current_figure(filename):
    plt.savefig(asset_path(filename), dpi=240, bbox_inches='tight')
    plt.close()
    print('Saved:', asset_path(filename))

def placeholder_figure(filename, title, message):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axis('off')
    ax.text(0.5, 0.62, title, ha='center', va='center', fontsize=18, fontweight='bold')
    ax.text(0.5, 0.42, message, ha='center', va='center', fontsize=12, wrap=True)
    save_current_figure(filename)

def setup_axis(ax, title, ylabel='Accuracy (%)'):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel(ylabel)
    ax.set_xticks(SNRS)
    ax.grid(True, alpha=0.25)

def metric_xy(metric, snr_filter=None):
    if not metric:
        return [], []
    xs = [snr for snr in SNRS if snr in metric]
    if snr_filter is not None:
        xs = [snr for snr in xs if snr_filter(snr)]
    ys = [100.0 * metric[snr] for snr in xs]
    return xs, ys

def draw_box(ax, x, y, w, h, text, fc='#f8fafc', ec='#334155', fontsize=10):
    patch = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.015', fc=fc, ec=ec, lw=1.3)
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fontsize, wrap=True)
    return patch

def draw_arrow(ax, start, end, color='#475569'):
    arrow = FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=14, lw=1.4, color=color)
    ax.add_patch(arrow)

def record_key(record):
    return (record['strategy'], record['model'], record['channel'])

def get_record(records, strategy, model, channel):
    for record in records:
        if record['strategy'] == strategy and record['model'] == model and record['channel'] == channel:
            return record
    return None


In [ ]:
# Load metrics and write inventory/summary tables
records = []
inventory_rows = []
summary_rows = []

for exp in EXPERIMENTS:
    exp_path = exp['path']
    acc_path = find_metric_file(exp_path, 'acc.pkl')
    acc_mod_path = find_metric_file(exp_path, 'acc_mod_snr.pkl')
    f1_path = find_metric_file(exp_path, 'f1_macro_scores.pkl')
    mcc_path = find_metric_file(exp_path, 'mcc_metric_scores.pkl')

    record = dict(exp)
    record['acc'] = normalize_metric_dict(load_pickle(acc_path))
    record['acc_mod_snr'] = load_pickle(acc_mod_path) if acc_mod_path else None
    record['f1'] = normalize_metric_dict(load_pickle(f1_path))
    record['mcc'] = normalize_metric_dict(load_pickle(mcc_path))
    record['weight_files'] = find_weight_files(exp_path)
    records.append(record)

    inventory_rows.append({
        'strategy': exp['strategy'],
        'model': exp['model'],
        'channel': exp['channel'],
        'path': str(exp_path),
        'acc_pkl': str(acc_path) if acc_path else 'MISSING',
        'acc_mod_snr_pkl': str(acc_mod_path) if acc_mod_path else 'MISSING',
        'f1_macro_scores_pkl': str(f1_path) if f1_path else 'MISSING',
        'mcc_metric_scores_pkl': str(mcc_path) if mcc_path else 'MISSING',
        'weight_files': ';'.join(record['weight_files']) if record['weight_files'] else 'MISSING',
    })

    acc = record['acc']
    f1 = record['f1']
    mcc = record['mcc']
    if acc:
        max_snr = max(acc, key=lambda snr: acc[snr])
        mean_acc = mean_for_snr(acc, lambda snr: True)
        low_acc = mean_for_snr(acc, lambda snr: snr <= 0)
        high_acc = mean_for_snr(acc, lambda snr: snr >= 0)
        acc_18 = acc.get(18, np.nan)
    else:
        max_snr = ''
        mean_acc = low_acc = high_acc = acc_18 = np.nan
    summary_rows.append({
        'strategy': exp['strategy'],
        'model': exp['model'],
        'channel': exp['channel'],
        'mean_accuracy_pct': safe_pct(mean_acc),
        'low_snr_accuracy_pct': safe_pct(low_acc),
        'high_snr_accuracy_pct': safe_pct(high_acc),
        'accuracy_at_18db_pct': safe_pct(acc_18),
        'max_accuracy_pct': safe_pct(acc[max_snr]) if acc else 'NA',
        'snr_at_max_accuracy': max_snr,
        'mean_f1_macro_pct': safe_pct(mean_for_snr(f1, lambda snr: True)) if f1 else 'NA',
        'mean_mcc_pct': safe_pct(mean_for_snr(mcc, lambda snr: True)) if mcc else 'NA',
    })

with open(asset_path('table_01_experiment_inventory.csv'), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(inventory_rows[0].keys()))
    writer.writeheader()
    writer.writerows(inventory_rows)

with open(asset_path('table_02_summary_metrics.csv'), 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print('Loaded records:', len(records))
print('Inventory CSV:', asset_path('table_01_experiment_inventory.csv'))
print('Summary CSV:', asset_path('table_02_summary_metrics.csv'))
for row in inventory_rows:
    missing = [key for key, value in row.items() if value == 'MISSING']
    if missing:
        print('Missing:', row['strategy'], row['model'], row['channel'], missing)


In [ ]:
# Figures 01-03: reference map, project pipeline, dataset/channel map
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Reference Baseline to Project Contributions', fontsize=17, fontweight='bold', pad=16)
draw_box(ax, 0.04, 0.57, 0.24, 0.24, 'Reference Paper\nZhang et al. 2022\nAMR models and datasets', '#e0f2fe')
draw_box(ax, 0.04, 0.19, 0.24, 0.24, 'Baseline Repo\nAMR-Benchmark\nLegacy Keras/TF1 style', '#dbeafe')
draw_box(ax, 0.38, 0.38, 0.24, 0.24, 'Our Starting Point\nAWGN baseline\nMCLDNN + PET-CGDNN', '#f8fafc')
draw_box(ax, 0.72, 0.68, 0.23, 0.18, 'TF2/Keras migration\nNamed layers + Colab support', '#dcfce7')
draw_box(ax, 0.72, 0.44, 0.23, 0.18, 'Rayleigh/Rician datasets\nSISO fading robustness', '#fef3c7')
draw_box(ax, 0.72, 0.20, 0.23, 0.18, 'Fine-tuning + metrics\nAccuracy, F1, MCC', '#fee2e2')
draw_arrow(ax, (0.28, 0.69), (0.38, 0.52))
draw_arrow(ax, (0.28, 0.31), (0.38, 0.47))
draw_arrow(ax, (0.62, 0.52), (0.72, 0.77))
draw_arrow(ax, (0.62, 0.50), (0.72, 0.53))
draw_arrow(ax, (0.62, 0.47), (0.72, 0.29))
save_current_figure('fig_01_reference_and_contribution_map.png')

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('End-to-End AMR Robustness Pipeline', fontsize=17, fontweight='bold', pad=16)
steps = [
    ('RadioML 2016.10a\nAWGN IQ samples', '#e0f2fe'),
    ('Synthetic SISO fading\nRayleigh, Rician K=3/K=10', '#fef3c7'),
    ('Model training\nMCLDNN + PET-CGDNN', '#dcfce7'),
    ('Fine-tuning\nAWGN pretrained to fading', '#ffedd5'),
    ('Evaluation\nAccuracy, F1 Macro, MCC', '#fee2e2'),
]
x_positions = [0.03, 0.225, 0.42, 0.615, 0.81]
for i, ((text, color), x) in enumerate(zip(steps, x_positions)):
    draw_box(ax, x, 0.36, 0.16, 0.28, text, color, fontsize=9)
    if i < len(steps) - 1:
        draw_arrow(ax, (x + 0.16, 0.50), (x_positions[i + 1], 0.50))
save_current_figure('fig_02_project_pipeline.png')

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.axis('off')
ax.set_title('Dataset Variants and Their Role in Experiments', fontsize=16, fontweight='bold', pad=14)
table_data = [
    ['Dataset', 'Channel model', 'Use in project', 'Expected effect'],
    ['RML2016.10a_dict.pkl', 'AWGN original', 'Baseline training + pretrained source', 'Reference benchmark'],
    ['RML2016.10a_rayleigh.pkl', 'Rayleigh NLOS', 'Fading training + fine-tuning target', 'Strong amplitude/phase distortion'],
    ['RML2016.10a_rician_K3.pkl', 'Rician K=3', 'Fading training + fine-tuning target', 'Moderate LOS'],
    ['RML2016.10a_rician_K10.pkl', 'Rician K=10', 'Fading training + fine-tuning target', 'Strong LOS, closer to AWGN'],
]
table = ax.table(cellText=table_data[1:], colLabels=table_data[0], loc='center', cellLoc='left', colLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#e5e7eb')
        cell.set_text_props(weight='bold')
    else:
        cell.set_facecolor('#ffffff' if row % 2 else '#f8fafc')
save_current_figure('fig_03_dataset_channel_map.png')


In [ ]:
# Figure 04: constellation examples from dataset files
def load_one_constellation(path, preferred_key=('QPSK', 18), sample_idx=0):
    if not path.exists():
        return None, 'dataset missing'
    data = load_pickle(path)
    if data is None:
        return None, 'could not load'
    key = preferred_key if preferred_key in data else sorted(data.keys())[0]
    sample = data[key][sample_idx]
    label = f'{key[0]} at {key[1]} dB'
    del data
    gc.collect()
    return sample, label

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle('Example IQ Constellations Across Channel Variants', fontsize=16, fontweight='bold')
for ax, (channel, path) in zip(axes.flat, DATASET_FILES.items()):
    sample, label = load_one_constellation(path)
    ax.set_title(channel, fontweight='bold')
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.grid(True, alpha=0.25)
    if sample is None:
        ax.text(0.5, 0.5, label, transform=ax.transAxes, ha='center', va='center')
        continue
    ax.scatter(sample[0], sample[1], s=14, alpha=0.75, color=CHANNEL_COLORS[channel])
    ax.text(0.03, 0.94, label, transform=ax.transAxes, fontsize=9, va='top')
    ax.axhline(0, color='#94a3b8', lw=0.7)
    ax.axvline(0, color='#94a3b8', lw=0.7)
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_current_figure('fig_04_channel_constellation_examples.png')


In [ ]:
# Figures 05-07: method and training strategy diagrams
fig, ax = plt.subplots(figsize=(12, 5.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Fading Dataset Generation Method', fontsize=16, fontweight='bold', pad=14)
draw_box(ax, 0.05, 0.55, 0.24, 0.24, 'IQ sample\nx = I + jQ\nshape: (2, 128)', '#e0f2fe')
draw_box(ax, 0.38, 0.62, 0.24, 0.18, 'Rayleigh\nh ~ CN(0, 1)\nNLOS fading', '#ede9fe')
draw_box(ax, 0.38, 0.30, 0.24, 0.22, 'Rician\nh = LOS + NLOS\nK = 3 or K = 10', '#fef3c7')
draw_box(ax, 0.73, 0.55, 0.22, 0.24, 'Faded signal\ny = h * x\nSaved as new pkl dataset', '#dcfce7')
draw_arrow(ax, (0.29, 0.67), (0.38, 0.71))
draw_arrow(ax, (0.29, 0.63), (0.38, 0.41))
draw_arrow(ax, (0.62, 0.71), (0.73, 0.69))
draw_arrow(ax, (0.62, 0.41), (0.73, 0.63))
ax.text(0.5, 0.13, 'Deterministic seeds make generated fading datasets reproducible: Rayleigh=2016, Rician K=3=3016, Rician K=10=4016.', ha='center', fontsize=10)
save_current_figure('fig_05_fading_generation_method.png')

fig, ax = plt.subplots(figsize=(12.5, 5.4))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Training Strategies Compared in the Presentation', fontsize=16, fontweight='bold', pad=14)
draw_box(ax, 0.05, 0.48, 0.24, 0.30, '1. AWGN baseline\nTrain and test on original AWGN RadioML\nReference performance', '#e0f2fe')
draw_box(ax, 0.38, 0.48, 0.24, 0.30, '2. Fading-trained\nTrain from scratch on Rayleigh/Rician datasets\nChannel-specific learning', '#dcfce7')
draw_box(ax, 0.71, 0.48, 0.24, 0.30, '3. Fine-tuned\nStart from AWGN pretrained weights\nAdapt to fading channels', '#ffedd5')
draw_arrow(ax, (0.29, 0.63), (0.38, 0.63))
draw_arrow(ax, (0.62, 0.63), (0.71, 0.63))
draw_box(ax, 0.20, 0.16, 0.60, 0.16, 'Evaluation: Accuracy for all experiments; F1 Macro and MCC where available for fine-tuned outputs.', '#f8fafc')
save_current_figure('fig_06_training_strategy_overview.png')

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Model Architecture Summary', fontsize=16, fontweight='bold', pad=14)
draw_box(ax, 0.08, 0.25, 0.36, 0.46, 'MCLDNN\nMulti-channel input\nConv1D/Conv2D feature extraction\nLSTM temporal modeling\nDense softmax classifier', '#dbeafe', fontsize=11)
draw_box(ax, 0.56, 0.25, 0.36, 0.46, 'PET-CGDNN\nLearnable phase estimation\nPhase transformation\nConv2D feature extraction\nGRU temporal modeling\nSoftmax classifier', '#fee2e2', fontsize=11)
ax.text(0.50, 0.13, 'Both models classify 11 modulation types from 128-sample I/Q sequences.', ha='center', fontsize=10)
save_current_figure('fig_07_model_architecture_summary.png')


In [ ]:
# Figures 08-10: core accuracy-vs-SNR comparisons
fig, ax = plt.subplots(figsize=(10, 5.5))
for model in MODELS:
    record = get_record(records, 'baseline_awgn', model, 'AWGN')
    if record and record['acc']:
        xs, ys = metric_xy(record['acc'])
        ax.plot(xs, ys, marker='o', lw=2.2, label=model, color=MODEL_COLORS[model])
setup_axis(ax, 'AWGN Baseline: MCLDNN vs PET-CGDNN')
ax.legend()
ax.set_ylim(0, 100)
save_current_figure('fig_08_awgn_baseline_accuracy_snr.png')

def plot_strategy_channel_grid(strategy, filename, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
    fig.suptitle(title, fontsize=16, fontweight='bold')
    for ax, channel in zip(axes, CHANNELS):
        plotted = False
        for model in MODELS:
            record = get_record(records, strategy, model, channel)
            if record and record['acc']:
                xs, ys = metric_xy(record['acc'])
                ax.plot(xs, ys, marker='o', lw=2, label=model, color=MODEL_COLORS[model])
                plotted = True
        setup_axis(ax, channel)
        ax.set_ylim(0, 100)
        if plotted:
            ax.legend(fontsize=9)
        else:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center', va='center')
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    save_current_figure(filename)

plot_strategy_channel_grid('fading_trained', 'fig_09_fading_trained_accuracy_snr.png', 'Models Trained From Scratch on Fading Datasets')
plot_strategy_channel_grid('fine_tuned', 'fig_10_finetuned_accuracy_snr.png', 'AWGN-Pretrained Models Fine-Tuned on Fading Datasets')


In [ ]:
# Figures 11-12: fine-tuning gain and fine-tuned F1/MCC summary
labels = []
gains = []
colors = []
for channel in CHANNELS:
    for model in MODELS:
        fading = get_record(records, 'fading_trained', model, channel)
        tuned = get_record(records, 'fine_tuned', model, channel)
        fading_mean = mean_for_snr(fading['acc'], lambda snr: True) if fading and fading['acc'] else np.nan
        tuned_mean = mean_for_snr(tuned['acc'], lambda snr: True) if tuned and tuned['acc'] else np.nan
        if not np.isnan(fading_mean) and not np.isnan(tuned_mean):
            labels.append(f'{model}\n{channel}')
            gains.append(100.0 * (tuned_mean - fading_mean))
            colors.append(MODEL_COLORS[model])

if gains:
    fig, ax = plt.subplots(figsize=(12, 5.5))
    bars = ax.bar(range(len(gains)), gains, color=colors, alpha=0.88)
    ax.axhline(0, color='#111827', lw=1)
    ax.set_title('Fine-Tuning Gain Over Fading-Trained Models', fontsize=15, fontweight='bold')
    ax.set_ylabel('Mean accuracy difference (percentage points)')
    ax.set_xticks(range(len(gains)))
    ax.set_xticklabels(labels, rotation=0, fontsize=9)
    ax.grid(True, axis='y', alpha=0.25)
    for bar, value in zip(bars, gains):
        ax.text(bar.get_x() + bar.get_width() / 2, value + (0.4 if value >= 0 else -1.2), f'{value:+.2f}', ha='center', va='bottom' if value >= 0 else 'top', fontsize=9)
    save_current_figure('fig_11_finetuning_gain_by_channel.png')
else:
    placeholder_figure('fig_11_finetuning_gain_by_channel.png', 'Fine-Tuning Gain', 'No paired fading-trained and fine-tuned accuracy files were found.')

ft_records = [record for record in records if record['strategy'] == 'fine_tuned']
ft_labels = [record['model'] + '\n' + record['channel'] for record in ft_records]
f1_values = [100.0 * mean_for_snr(record['f1'], lambda snr: True) if record['f1'] else np.nan for record in ft_records]
mcc_values = [100.0 * mean_for_snr(record['mcc'], lambda snr: True) if record['mcc'] else np.nan for record in ft_records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for ax, values, title in zip(axes, [f1_values, mcc_values], ['Fine-Tuned F1 Macro', 'Fine-Tuned MCC']):
    x = np.arange(len(ft_records))
    bar_colors = [MODEL_COLORS[record['model']] for record in ft_records]
    ax.bar(x, values, color=bar_colors, alpha=0.88)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ft_labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('Mean score (%)')
    ax.grid(True, axis='y', alpha=0.25)
    for i, value in enumerate(values):
        text = 'NA' if np.isnan(value) else f'{value:.1f}'
        ax.text(i, 2 if np.isnan(value) else value + 1.0, text, ha='center', fontsize=8)
axes[0].set_ylim(0, 100)
plt.tight_layout()
save_current_figure('fig_12_finetuned_f1_mcc_summary.png')


In [ ]:
# Figures 13-14: low-SNR robustness and model/channel heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
fig.suptitle('Low-SNR Robustness Focus (-20 dB to 0 dB)', fontsize=16, fontweight='bold')
for ax, channel in zip(axes, CHANNELS):
    for strategy, linestyle in [('fading_trained', '-'), ('fine_tuned', '--')]:
        for model in MODELS:
            record = get_record(records, strategy, model, channel)
            if record and record['acc']:
                xs, ys = metric_xy(record['acc'], snr_filter=lambda snr: snr <= 0)
                label = f'{model} {STRATEGY_LABELS[strategy]}'
                ax.plot(xs, ys, marker='o', lw=1.8, ls=linestyle, color=MODEL_COLORS[model], label=label)
    setup_axis(ax, channel)
    ax.set_ylim(0, 100)
    ax.legend(fontsize=7)
plt.tight_layout(rect=[0, 0, 1, 0.92])
save_current_figure('fig_13_low_snr_robustness.png')

heatmap_rows = []
heatmap_labels = []
for strategy in ['baseline_awgn', 'fading_trained', 'fine_tuned']:
    for model in MODELS:
        row = []
        for channel in ALL_CHANNELS:
            record = get_record(records, strategy, model, channel)
            row.append(100.0 * mean_for_snr(record['acc'], lambda snr: True) if record and record['acc'] else np.nan)
        heatmap_rows.append(row)
        heatmap_labels.append(f'{model}\n{STRATEGY_LABELS[strategy]}')
heatmap = np.array(heatmap_rows, dtype=float)
masked = np.ma.masked_invalid(heatmap)
cmap = plt.cm.viridis.copy()
cmap.set_bad('#e5e7eb')
fig, ax = plt.subplots(figsize=(10.5, 7))
im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=100)
ax.set_title('Mean Accuracy Heatmap by Model, Strategy, and Channel', fontsize=15, fontweight='bold')
ax.set_xticks(np.arange(len(ALL_CHANNELS)))
ax.set_xticklabels(ALL_CHANNELS)
ax.set_yticks(np.arange(len(heatmap_labels)))
ax.set_yticklabels(heatmap_labels)
for i in range(heatmap.shape[0]):
    for j in range(heatmap.shape[1]):
        text = 'NA' if np.isnan(heatmap[i, j]) else f'{heatmap[i, j]:.1f}'
        ax.text(j, i, text, ha='center', va='center', color='white' if not np.isnan(heatmap[i, j]) and heatmap[i, j] < 55 else '#111827', fontsize=9)
fig.colorbar(im, ax=ax, label='Mean accuracy (%)')
plt.tight_layout()
save_current_figure('fig_14_model_channel_heatmap.png')


In [ ]:
# Figures 15-16: best-model summary panel and conclusion takeaways
def best_record_for(channel, strategies):
    candidates = []
    for strategy in strategies:
        for model in MODELS:
            record = get_record(records, strategy, model, channel)
            if record and record['acc']:
                candidates.append((mean_for_snr(record['acc'], lambda snr: True), record))
    if not candidates:
        return None, np.nan
    return max(candidates, key=lambda item: item[0])[1], max(candidates, key=lambda item: item[0])[0]

cards = []
awgn_best, awgn_score = best_record_for('AWGN', ['baseline_awgn'])
if awgn_best:
    cards.append(('Best AWGN baseline', awgn_best['model'] + f'\nMean accuracy: {100 * awgn_score:.2f}%'))
for channel in CHANNELS:
    best, score = best_record_for(channel, ['fading_trained', 'fine_tuned'])
    if best:
        body = best['model'] + '\n' + STRATEGY_LABELS[best['strategy']] + f'\nMean accuracy: {100 * score:.2f}%'
        cards.append((f'Best on {channel}', body))

low_candidates = []
for record in records:
    if record['acc']:
        low_candidates.append((mean_for_snr(record['acc'], lambda snr: snr <= 0), record))
if low_candidates:
    best_low_score, best_low = max(low_candidates, key=lambda item: item[0])
    body = best_low['model'] + '\n' + best_low['channel'] + ' / ' + STRATEGY_LABELS[best_low['strategy']] + f'\nLow-SNR mean: {100 * best_low_score:.2f}%'
    cards.append(('Best low-SNR mean', body))

fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Best Model and Channel Summary', fontsize=17, fontweight='bold', pad=14)
positions = [(0.04, 0.58), (0.37, 0.58), (0.70, 0.58), (0.20, 0.22), (0.54, 0.22)]
for (title, body), (x, y) in zip(cards[:5], positions):
    draw_box(ax, x, y, 0.27, 0.24, f'{title}\n\n{body}', '#f8fafc', '#334155', fontsize=10)
save_current_figure('fig_15_best_model_summary_panel.png')

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Final Takeaways for the Presentation', fontsize=17, fontweight='bold', pad=14)
takeaways = [
    ('AWGN is not enough', 'A single AWGN benchmark does not represent channel robustness under realistic fading.'),
    ('Fading datasets add realism', 'Rayleigh and Rician variants expose amplitude and phase distortion effects.'),
    ('Fine-tuning is practical', 'AWGN-pretrained models can be adapted to channel-specific fading conditions.'),
    ('Low SNR remains hard', 'Performance remains most fragile from -20 dB to 0 dB across strategies.'),
]
for i, (title, body) in enumerate(takeaways):
    x = 0.06 + (i % 2) * 0.47
    y = 0.56 if i < 2 else 0.22
    draw_box(ax, x, y, 0.40, 0.22, f'{title}\n\n{body}', '#eef2ff' if i % 2 == 0 else '#ecfdf5', '#334155', fontsize=10)
save_current_figure('fig_16_final_conclusion_takeaways.png')


## Expected output files

The notebook writes these assets to `presentation_assets/`:

- `fig_01_reference_and_contribution_map.png`
- `fig_02_project_pipeline.png`
- `fig_03_dataset_channel_map.png`
- `fig_04_channel_constellation_examples.png`
- `fig_05_fading_generation_method.png`
- `fig_06_training_strategy_overview.png`
- `fig_07_model_architecture_summary.png`
- `fig_08_awgn_baseline_accuracy_snr.png`
- `fig_09_fading_trained_accuracy_snr.png`
- `fig_10_finetuned_accuracy_snr.png`
- `fig_11_finetuning_gain_by_channel.png`
- `fig_12_finetuned_f1_mcc_summary.png`
- `fig_13_low_snr_robustness.png`
- `fig_14_model_channel_heatmap.png`
- `fig_15_best_model_summary_panel.png`
- `fig_16_final_conclusion_takeaways.png`
- `table_01_experiment_inventory.csv`
- `table_02_summary_metrics.csv`